# 00 – Setup & explore candidate games

Goal: download the candidate ARC-AGI-3 games, look at them, and **export their source code + first frames** so we can read the real rules and write an automatic solver.

Run top to bottom on JupyterHub. At the end, download `export_for_claude.zip` and put it in your `project multimodal` folder.

In [ ]:
%pip install -q "arc-agi==0.9.9" pillow matplotlib

## API key
Only 3 games (ls20, ft09, vc33) are free without a key. For G50T get a free key at https://arcprize.org/api-keys (log in → create key) and paste it into `arc_api_key.txt` next to this notebook. **Don't share the key with anyone or paste it into chat.**

In [ ]:
import os
from pathlib import Path
key_file = Path("arc_api_key.txt")
if key_file.exists():
    os.environ["ARC_API_KEY"] = key_file.read_text().strip()
    print("ARC API key loaded")
else:
    print("No arc_api_key.txt found - only ls20/ft09/vc33 will download")

In [ ]:
import arc_agi, json, shutil, copy, time
import numpy as np
from arc_agi import OperationMode
from arc_agi.rendering import frame_to_rgb_array
from PIL import Image

ENV_DIR = "./environment_files"
CANDIDATES = ["g50t"]   # our game (group ARC3_G50T)

arcade = arc_agi.Arcade(operation_mode=OperationMode.NORMAL, environments_dir=ENV_DIR)

def render(obs, scale=8):
    return Image.fromarray(frame_to_rgb_array(steps=0, frame=obs.frame[-1], scale=scale))

## Inspect each candidate

In [ ]:
OUT = Path("export_for_claude"); OUT.mkdir(exist_ok=True)
report = {}
for gid in CANDIDATES:
    try:
        env = arcade.make(gid)
        obs = env.reset()
    except Exception as e:
        print(f"{gid}: could not load -> {e!r}")
        report[gid] = {"error": repr(e)}
        continue
    info = {
        "actions": [a.name for a in env.action_space],
        "available_actions": [str(a) for a in obs.available_actions],
        "win_levels": obs.win_levels,
        "frame_shape": list(np.array(obs.frame[-1]).shape),
        "colors_used": np.unique(obs.frame[-1]).tolist(),
    }
    # can we copy the game state? (makes the solver MUCH faster)
    try:
        t = time.time(); env2 = copy.deepcopy(env); info["deepcopy_ok"] = True
        info["deepcopy_seconds"] = round(time.time() - t, 4)
        before = np.array(env.step(env.action_space[0]).frame[-1])
        after_copy = np.array(env2.step(env2.action_space[0]).frame[-1])
        info["deepcopy_independent_and_same"] = bool((before == after_copy).all())
    except Exception as e:
        info["deepcopy_ok"] = False; info["deepcopy_error"] = repr(e)
    report[gid] = info
    render(env.reset()).save(OUT / f"{gid}_start.png")
    np.save(OUT / f"{gid}_start.npy", np.array(env.reset().frame[-1]))
    print(gid, json.dumps(info, indent=1))
    display(render(env.reset(), scale=6))

(OUT / "report.json").write_text(json.dumps(report, indent=2))

## Try one action of each type and record what changes

In [ ]:
for gid in CANDIDATES:
    if "error" in report.get(gid, {}): continue
    env = arcade.make(gid)
    for a in env.action_space:
        obs0 = env.reset(); f0 = np.array(obs0.frame[-1])
        try:
            obs1 = env.step(a)
        except Exception as e:
            print(gid, a.name, "error:", repr(e)); continue
        f1 = np.array(obs1.frame[-1])
        diff = np.argwhere(f0 != f1)
        print(f"{gid} {a.name}: {len(diff)} cells changed, frames returned={len(obs1.frame)}, state={obs1.state.name}")
        render(obs1).save(OUT / f"{gid}_after_{a.name}.png")

## Export the game source code + zip everything

In [ ]:
for gid in CANDIDATES:
    src = Path(ENV_DIR) / gid
    if src.exists():
        shutil.copytree(src, OUT / "source" / gid, dirs_exist_ok=True)
shutil.make_archive("export_for_claude", "zip", OUT)
print("Done -> download export_for_claude.zip (right-click in the file browser -> Download)")